In [1]:
# !pip install pymodbus

In [1]:
import pymodbus
from pymodbus.client import ModbusTcpClient
client = ModbusTcpClient('210.119.14.56', port=502)

client.connect()

True

In [2]:
# 컴베어, 이미터, 리무버 초기화
# client.write_coils(0, [0] * 8)
# client.write_coils(0, [1] * 8)

In [3]:
# 컨베이어, 이미터, 리무버 초기화


client.write_coil(0, 0)
# client.write_coil(5, 0)
# client.write_coil(6, 0)
# client.write_coil(7, 0)
client.write_coil(20, 0)
client.write_coil(21, 0)
client.write_coil(22, 0)

In [4]:
import time as tt
import threading

# 컨베어, 이미터, 리무버 켜기
client.write_coil(0, 1)
# client.write_coil(5, 1)
# client.write_coil(6, 1)
# client.write_coil(7, 1)
client.write_coil(20, 1)
client.write_coil(21, 1)
client.write_coil(22, 1)
count = [0] # 팁   

In [64]:
running = True

def scan():
    while running:
        sensor = client.read_discrete_inputs(0, count=8, device_id=1).bits[0] # 옛 slave, unit_id
        if sensor:
            client.write_coil(1,1) # 스토퍼 ON
            tt.sleep(3)
            client.write_coil(1,0)
            tt.sleep(3)
        else:    
            client.write_coil(1,0) # 스토퍼 OFF
        tt.sleep(0.1)
    print("스캔종료.")
th_stopper = threading.Thread(target=scan, daemon=True)
print("쓰레드 시작")
th_stopper.start()

쓰레드 시작


In [65]:
# runnging = True
# th_stopper = threading.Thread(target=scan, daemon=True)
# print("쓰레드 시작")
# th_stopper.start()

In [72]:
pusher_run = True

def pusher():
    busy = False # 플래그
    while pusher_run:
        sensor2 = client.read_discrete_inputs(0, count=8, device_id=1).bits[1] # 옛 slave, unit_id
        if sensor2 and not busy:
            busy = True
            tt.sleep(1.5)
            client.write_coil(2, 1) # 푸셔 ON
            count[0] += 1
            client.write_register(0,count[0])
        if client.read_discrete_inputs(0, count=8, device_id=1).bits[3]:
            client.write_coil(2, 0) # 푸셔 OFF
            busy = False
        tt.sleep(0.1)
    print("푸셔스캔종료.")
th_pusher = threading.Thread(target=pusher, daemon=True)
print("푸셔쓰레드 시작")
th_pusher.start()

푸셔쓰레드 시작


In [67]:
# pusher_run = False

In [68]:
HR = client.read_holding_registers(0, count=4).registers

In [69]:
HR

[1911, 521, 1908, 0]